# MatNet Covertype Comparison

This notebook adapts the experiment to the **UCI Covertype** dataset while keeping the existing hidden-layer architectures unchanged and only changing the final output size to match the dataset.

It does four things:

1. Loads a **20k-row Covertype subsample** with 54 input features and 7 classes.
2. Keeps the preselected MatNet hidden-layer layouts for each matrix size `n in {1, 2, 4, 8, 16, 32}`.
3. Trains all selected models plus a traditional MLP baseline.
4. Reports accuracy, parameter counts, and compute estimates.


In [1]:
from __future__ import annotations

import itertools
import math
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
from flax import linen as nn

ROOT = Path.cwd().resolve()
if not (ROOT / 'matnet').exists() and (ROOT.parent / 'matnet').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from matnet.datasets import load_covertype
from matnet.models.builder import MatrixNetwork


## Dataset

Covertype has **54 input features** and **7 forest cover classes**. To keep the notebook runnable, this version uses a stratified subsample while preserving the original hidden-layer architectures and only changing the output layer size.


In [2]:
COVERTYPE_SOURCE = 'https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz'
COVERTYPE_SAMPLE_SIZE = 20_000

dataset = load_covertype(
    COVERTYPE_SOURCE,
    sample_size=COVERTYPE_SAMPLE_SIZE,
    test_size=0.2,
)

X_train = dataset.X_train
X_test = dataset.X_test
y_train = dataset.y_train
y_test = dataset.y_test
class_names = np.array(dataset.class_names)

num_classes = dataset.output_dim
input_dim = dataset.input_dim

print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
print(f'Input dim: {input_dim}')
print(f'Classes: {class_names}')


Train shape: (16000, 54)
Test shape: (4000, 54)
Input dim: 54
Classes: ['Spruce/Fir' 'Lodgepole Pine' 'Ponderosa Pine' 'Cottonwood/Willow'
 'Aspen' 'Douglas-fir' 'Krummholz']


## Traditional Baseline

To make the **traditional MLP** exactly match the selected **MatNet `n=1`** parameter count, it includes an initial width-`1` dense layer. That mirrors the `InputScaling` stage of MatNet when `n = 1`.

Normalization is disabled for the sweep so the `n=1` MatNet and traditional baseline can be matched exactly.

In [3]:
class TraditionalMLP(nn.Module):
    hidden_dims: tuple[int, ...]
    output_dim: int
    dtype: jnp.dtype = jnp.float32

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        for width in self.hidden_dims:
            x = nn.Dense(width, dtype=self.dtype)(x)
            x = nn.relu(x)
        x = nn.Dense(self.output_dim, dtype=self.dtype)(x)
        return x

## Parameter Counting And Architecture Metadata

Architectures are now selected automatically with a **fast constrained search** that:

- uses **powers-of-two widths only**,
- allows only **monotonic** or **rise-then-fall** layouts (no decrease-then-rise valleys),
- targets a parameter budget of **50K to 75K** (midpoint 62.5K).

Budget filtering uses feature-aware input accounting, and final reported parameter counts use **actual JAX initialization** of each selected model.

In [14]:
TARGET_MIN = 50_000
TARGET_MAX = 75_000
TARGET_MID = 62_500
N_VALUES = [1, 2, 4, 8, 16, 32]

# How to account for input cost during architecture search:
# - 'matrix': exact InputScaling params = input_dim * n^2 + n^2
# - 'feature_only': account inputs without n^2 multiplication
SEARCH_INPUT_ACCOUNTING = 'feature_only'

# Powers-of-two width candidates only, tuned per n for a fast search.
POWER_WIDTHS = {
    1:  [8, 16, 32, 64, 128],
    2:  [8, 16, 32, 64, 128],
    4:  [4, 8, 16, 32, 64, 128],
    8:  [2, 4, 8, 16, 32, 64],
    16: [1, 2, 4, 8, 16, 32],
    32: [1, 2, 4, 8, 16],
}


def count_params(variables) -> int:
    return int(sum(x.size for x in jax.tree_util.tree_leaves(variables)))


def count_matnet_params_budget(
    n: int,
    hidden_dims: tuple[int, ...],
    *,
    use_bias: bool = True,
    input_accounting: str = 'feature_only',
) -> int:
    """Fast closed-form count used only for architecture search budgeting."""
    n2 = n * n

    if input_accounting == 'matrix':
        total = input_dim * n2 + n2
    elif input_accounting == 'feature_only':
        total = input_dim
    else:
        raise ValueError(f'Unknown input_accounting mode: {input_accounting}')

    current_dim = 1
    for hidden_dim in hidden_dims:
        total += hidden_dim * current_dim * n2
        if use_bias:
            total += hidden_dim * n2
        current_dim = hidden_dim

    total += current_dim * n2 * num_classes
    if use_bias:
        total += num_classes

    return int(total)


def count_matnet_params_init_jax(n: int, hidden_dims: tuple[int, ...], *, use_bias: bool = True) -> int:
    """True parameter count from actual JAX/Flax initialization."""
    model = MatrixNetwork(
        matrix_size=n,
        hidden_dims=hidden_dims,
        output_dim=num_classes,
        use_bias=use_bias,
        use_input_scaling=True,
        use_normalization=False,
    )
    params = model.init(jax.random.PRNGKey(0), jnp.ones((1, input_dim), dtype=jnp.float32))
    return count_params(params)


def count_traditional_params(hidden_dims: tuple[int, ...]) -> int:
    """True parameter count from actual JAX/Flax initialization."""
    model = TraditionalMLP(hidden_dims=hidden_dims, output_dim=num_classes)
    params = model.init(jax.random.PRNGKey(0), jnp.ones((1, input_dim), dtype=jnp.float32))
    return count_params(params)


def generate_power_architectures(widths: list[int]) -> list[tuple[int, ...]]:
    archs: set[tuple[int, ...]] = set()

    # 1-layer options (kept only as a last fallback).
    for a in widths:
        archs.add((a,))

    for i, a in enumerate(widths):
        for b in widths[i + 1:]:
            # Monotonic increasing/decreasing
            archs.add((a, b))
            archs.add((b, a))

            # Rise then fall (peak) is allowed
            archs.add((a, b, a))

            for c in widths:
                if c <= b:
                    continue

                # Strictly monotonic
                archs.add((a, b, c))
                archs.add((c, b, a))

                # Peak-style 4-layer
                archs.add((a, b, c, b))

    return sorted(archs, key=lambda x: (len(x), x))


def is_valley_pattern(hidden_dims: tuple[int, ...]) -> bool:
    """Valley means it decreases and then increases (disallowed)."""
    if len(hidden_dims) < 3:
        return False
    diffs = [b - a for a, b in zip(hidden_dims[:-1], hidden_dims[1:])]
    seen_down = False
    for d in diffs:
        if d < 0:
            seen_down = True
        if seen_down and d > 0:
            return True
    return False


def architecture_style_rank(hidden_dims: tuple[int, ...]) -> int:
    """Lower is better: peak/monotonic, singleton fallback, valley worst (should not occur)."""
    if is_valley_pattern(hidden_dims):
        return 3
    if len(hidden_dims) < 2:
        return 2

    diffs = [b - a for a, b in zip(hidden_dims[:-1], hidden_dims[1:])]
    has_up = any(d > 0 for d in diffs)
    has_down = any(d < 0 for d in diffs)

    if has_up and has_down:
        return 0  # rise then fall (peak)
    if has_up or has_down:
        return 1  # monotonic
    return 2


def search_matnet_architecture(n: int) -> tuple[tuple[int, ...], int]:
    candidates = generate_power_architectures(POWER_WIDTHS[n])
    best = None

    for hidden_dims in candidates:
        if is_valley_pattern(hidden_dims):
            continue

        budget_param_count = count_matnet_params_budget(
            n,
            hidden_dims,
            input_accounting=SEARCH_INPUT_ACCOUNTING,
        )
        in_budget = TARGET_MIN <= budget_param_count <= TARGET_MAX

        score = (
            0 if in_budget else 1,
            architecture_style_rank(hidden_dims),
            abs(budget_param_count - TARGET_MID),
            -len(hidden_dims),
            hidden_dims,
        )

        if best is None or score < best[0]:
            best = (score, hidden_dims, budget_param_count)

    return best[1], best[2]


selected_architectures = {}
for n in N_VALUES:
    hidden_dims, budget_param_count = search_matnet_architecture(n)
    true_param_count = count_matnet_params_init_jax(n, hidden_dims)
    selected_architectures[n] = {
        'hidden_dims': hidden_dims,
        'budget_param_count': budget_param_count,
        'param_count': true_param_count,
    }


arch_df = pd.DataFrame(
    [
        {
            'model': f'MatNet n={n}',
            'n': n,
            'hidden_dims': selected_architectures[n]['hidden_dims'],
            'budget_param_count': selected_architectures[n]['budget_param_count'],
            'param_count': selected_architectures[n]['param_count'],
        }
        for n in N_VALUES
    ]
)
arch_df

,model,n,hidden_dims,budget_param_count,param_count
0,MatNet n=1,1,"(32, 64, 128, 64)",19261,19262
1,MatNet n=2,2,"(64, 128, 64)",68669,68835
2,MatNet n=4,4,"(16, 128, 16)",70205,71031
3,MatNet n=8,8,"(8, 64, 8)",74813,78279
4,MatNet n=16,16,"(16, 8, 4)",59453,73479
5,MatNet n=32,32,"(2, 8, 2)",61501,117767


## Traditional Baseline Matched To MatNet `n=1`

Because `InputScaling` becomes a dense projection to width `1` when `n=1`, the exact traditional match is:

`traditional_hidden_dims = (1, *matnet_n1_hidden_dims)`

In [13]:
matnet_n1_hidden_dims = selected_architectures[1]['hidden_dims']
traditional_hidden_dims = (1, *matnet_n1_hidden_dims)

traditional_param_count = count_traditional_params(traditional_hidden_dims)
matnet_n1_param_count = selected_architectures[1]['param_count']

print('MatNet n=1 hidden dims:', matnet_n1_hidden_dims)
print('Traditional hidden dims:', traditional_hidden_dims)
print('MatNet n=1 params:', matnet_n1_param_count)
print('Traditional params:', traditional_param_count)

assert traditional_param_count == matnet_n1_param_count, 'Traditional baseline must match MatNet n=1 exactly.'

MatNet n=1 hidden dims: (32, 64, 128, 64)
Traditional hidden dims: (1, 32, 64, 128, 64)
MatNet n=1 params: 19262
Traditional params: 19262


## Training Utilities

In [ ]:
BATCH_SIZE = 512
EPOCHS = 10
LEARNING_RATE_TRADITIONAL = 0.001
LEARNING_RATE_MATNET = 0.001
GRAD_CLIP_NORM = 1.01
ADAM_MULTS_PER_PARAM = 7

X_train_jnp = jnp.asarray(X_train)
X_test_jnp = jnp.asarray(X_test)
y_train_jnp = jnp.asarray(y_train)
y_test_jnp = jnp.asarray(y_test)

NUM_TRAIN_SAMPLES = int(X_train_jnp.shape[0])
NUM_TEST_SAMPLES = int(X_test_jnp.shape[0])
NUM_TRAIN_BATCHES = math.ceil(NUM_TRAIN_SAMPLES / BATCH_SIZE)


def accuracy(logits: jnp.ndarray, labels: jnp.ndarray) -> jnp.ndarray:
    preds = jnp.argmax(logits, axis=-1)
    return jnp.mean((preds == labels).astype(jnp.float32))


def matnet_forward_mults_per_sample(
    *,
    input_dim: int,
    matrix_size: int,
    hidden_dims: tuple[int, ...],
    output_dim: int,
    use_input_scaling: bool = True,
) -> int:
    n2 = matrix_size * matrix_size
    total = 0

    if use_input_scaling:
        total += input_dim * n2
        current_dim = 1
    else:
        current_dim = 1

    for hidden_dim in hidden_dims:
        total += hidden_dim * current_dim * n2
        current_dim = hidden_dim

    total += current_dim * n2 * output_dim
    return int(total)


def traditional_forward_mults_per_sample(*, input_dim: int, hidden_dims: tuple[int, ...], output_dim: int) -> int:
    dims = (input_dim, *hidden_dims, output_dim)
    return int(sum(in_dim * out_dim for in_dim, out_dim in zip(dims[:-1], dims[1:])))


def estimate_total_training_mults(*, forward_mults_per_sample: int, param_count: int) -> int:
    forward_per_epoch = NUM_TRAIN_SAMPLES * forward_mults_per_sample
    backward_per_epoch = 2 * forward_per_epoch
    optimizer_per_epoch = NUM_TRAIN_BATCHES * param_count * ADAM_MULTS_PER_PARAM
    return int(EPOCHS * (forward_per_epoch + backward_per_epoch + optimizer_per_epoch))


def estimate_total_inference_mults(*, forward_mults_per_sample: int, num_samples: int) -> int:
    return int(num_samples * forward_mults_per_sample)


def make_train_step(model, optimizer):
    @jax.jit
    def train_step(params, opt_state, x_batch, y_batch):
        def loss_fn(p):
            logits = model.apply(p, x_batch)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits, y_batch).mean()
            return loss, logits

        (loss, logits), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        grad_norm = optax.global_norm(grads)
        grads = jax.tree_util.tree_map(
            lambda g: jnp.clip(g, -GRAD_CLIP_NORM, GRAD_CLIP_NORM),
            grads,
        )
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        acc = accuracy(logits, y_batch)
        return params, opt_state, loss, acc, grad_norm

    return train_step


def make_eval_step(model):
    @jax.jit
    def eval_step(params, x_batch, y_batch):
        logits = model.apply(params, x_batch)
        loss = optax.softmax_cross_entropy_with_integer_labels(logits, y_batch).mean()
        acc = accuracy(logits, y_batch)
        return loss, acc

    return eval_step


def iterate_minibatches(x, y, batch_size, rng):
    indices = jax.random.permutation(rng, x.shape[0])
    for start in range(0, x.shape[0], batch_size):
        batch_idx = indices[start:start + batch_size]
        yield x[batch_idx], y[batch_idx]


def train_model(model, x_train, y_train, x_test, y_test, *, epochs=EPOCHS, learning_rate=LEARNING_RATE_MATNET):
    rng = jax.random.PRNGKey(0)
    params = model.init(rng, x_train[:1])
    optimizer = optax.adam(learning_rate)
    opt_state = optimizer.init(params)
    train_step = make_train_step(model, optimizer)
    eval_step = make_eval_step(model)

    history = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': [],
        'grad_norm': [],
    }

    for epoch in range(epochs):
        rng, epoch_rng = jax.random.split(rng)
        batch_losses = []
        batch_accs = []
        batch_grad_norms = []
        for x_batch, y_batch in iterate_minibatches(x_train, y_train, BATCH_SIZE, epoch_rng):
            params, opt_state, loss, acc, grad_norm = train_step(params, opt_state, x_batch, y_batch)
            batch_losses.append(float(loss))
            batch_accs.append(float(acc))
            batch_grad_norms.append(float(grad_norm))

        test_loss, test_acc = eval_step(params, x_test, y_test)
        history['train_loss'].append(float(np.mean(batch_losses)))
        history['train_acc'].append(float(np.mean(batch_accs)))
        history['test_loss'].append(float(test_loss))
        history['test_acc'].append(float(test_acc))
        history['grad_norm'].append(float(np.mean(batch_grad_norms)))

        if not np.isfinite(history['train_loss'][-1]) or not np.isfinite(history['test_loss'][-1]):
            raise ValueError('Non-finite loss encountered. Reduce learning rate further.')

    return params, history


## Build And Train All Models

In [ ]:
models = {}
models['Traditional'] = {
    'model': TraditionalMLP(hidden_dims=traditional_hidden_dims, output_dim=num_classes),
    'learning_rate': LEARNING_RATE_TRADITIONAL,
}

for n in N_VALUES:
    models[f'MatNet n={n}'] = {
        'model': MatrixNetwork(
            matrix_size=n,
            hidden_dims=selected_architectures[n]['hidden_dims'],
            output_dim=num_classes,
            use_bias=True,
            use_input_scaling=True,
            use_normalization=False,
        ),
        'learning_rate': LEARNING_RATE_MATNET,
    }

results = []
histories = {}
trained_params = {}

for name, spec in models.items():
    model = spec['model']
    learning_rate = spec['learning_rate']
    print(f'Training {name} with learning_rate={learning_rate} ...')
    params, history = train_model(
        model,
        X_train_jnp,
        y_train_jnp,
        X_test_jnp,
        y_test_jnp,
        learning_rate=learning_rate,
    )
    trained_params[name] = params
    histories[name] = history

    param_count = count_params(params)
    final_test_acc = history['test_acc'][-1]
    final_test_loss = history['test_loss'][-1]

    if name == 'Traditional':
        architecture = traditional_hidden_dims
        n_value = 'dense'
        forward_mults_per_sample = traditional_forward_mults_per_sample(
            input_dim=input_dim,
            hidden_dims=traditional_hidden_dims,
            output_dim=num_classes,
        )
    else:
        n_value = int(name.split('=')[1])
        architecture = selected_architectures[n_value]['hidden_dims']
        forward_mults_per_sample = matnet_forward_mults_per_sample(
            input_dim=input_dim,
            matrix_size=n_value,
            hidden_dims=architecture,
            output_dim=num_classes,
            use_input_scaling=True,
        )

    training_mults_est = estimate_total_training_mults(
        forward_mults_per_sample=forward_mults_per_sample,
        param_count=param_count,
    )
    inference_mults_test_est = estimate_total_inference_mults(
        forward_mults_per_sample=forward_mults_per_sample,
        num_samples=NUM_TEST_SAMPLES,
    )

    results.append({
        'model': name,
        'n': n_value,
        'hidden_dims': architecture,
        'param_count': param_count,
        'learning_rate': learning_rate,
        'forward_mults_per_sample': forward_mults_per_sample,
        'total_training_mults_est': training_mults_est,
        'test_inference_mults_est': inference_mults_test_est,
        'test_accuracy': final_test_acc,
        'test_loss': final_test_loss,
    })

results_df = pd.DataFrame(results).sort_values('model').reset_index(drop=True)
results_df


## Sanity Check: MatNet `n=1` And Traditional Match In Parameter Count

In [ ]:
trad_params = int(results_df.loc[results_df['model'] == 'Traditional', 'param_count'].iloc[0])
matnet_n1_params = int(results_df.loc[results_df['model'] == 'MatNet n=1', 'param_count'].iloc[0])
print('Traditional params:', trad_params)
print('MatNet n=1 params:', matnet_n1_params)
assert trad_params == matnet_n1_params

## Plot 1: Parameter Counts

In [7]:
plot_df = results_df.copy()
plt.figure(figsize=(12, 5))
plt.bar(plot_df['model'], plot_df['param_count'], color='steelblue')
plt.axhline(TARGET_MIN, color='darkgreen', linestyle='--', label='50k lower bound')
plt.axhline(TARGET_MAX, color='darkred', linestyle='--', label='75k upper bound')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Parameter count')
plt.title('Parameter count by model')
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'results_df' is not defined

## Plot 2: Test Accuracy

In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(plot_df['model'], plot_df['test_accuracy'], color='darkorange')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Test accuracy')
plt.ylim(0.0, 1.0)
plt.title('Final test accuracy by model')
plt.tight_layout()
plt.show()

## Plot 3: Parameter Count vs Accuracy

In [ ]:
plt.figure(figsize=(8, 6))
for _, row in plot_df.iterrows():
    plt.scatter(row['param_count'], row['test_accuracy'], s=120)
    plt.text(row['param_count'] + 150, row['test_accuracy'], row['model'], fontsize=9)
plt.xlabel('Parameter count')
plt.ylabel('Test accuracy')
plt.title('Accuracy vs parameter count')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 4: Test Accuracy Curves

In [ ]:
plt.figure(figsize=(12, 6))
for name, history in histories.items():
    plt.plot(history['test_acc'], label=name)
plt.xlabel('Epoch')
plt.ylabel('Test accuracy')
plt.ylim(0.0, 1.0)
plt.title('Test accuracy across training')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Plot 5: Estimated Training Multiplications


In [ ]:
plt.figure(figsize=(12, 5))
plot_df = results_df.copy()
plt.bar(plot_df['model'], plot_df['total_training_mults_est'], color='seagreen')
plt.yscale('log')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Estimated total training multiplications (log scale)')
plt.title('Estimated multiplication count for full training run')
plt.tight_layout()
plt.show()


## Plot 6: Estimated Test Inference Multiplications


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(plot_df['model'], plot_df['test_inference_mults_est'], color='slateblue')
plt.yscale('log')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Estimated test-set inference multiplications (log scale)')
plt.title('Estimated multiplication count for one inference pass over the test set')
plt.tight_layout()
plt.show()


## Plot 7: Inference Multiplications vs Accuracy


In [ ]:
plt.figure(figsize=(8, 6))
for _, row in plot_df.iterrows():
    plt.scatter(row['test_inference_mults_est'], row['test_accuracy'], s=120)
    plt.text(row['test_inference_mults_est'] * 1.02, row['test_accuracy'], row['model'], fontsize=9)
plt.xscale('log')
plt.xlabel('Estimated test-set inference multiplications')
plt.ylabel('Test accuracy')
plt.title('Accuracy vs inference multiplication count')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Result Table Sorted By Accuracy


In [ ]:
results_df.sort_values(
    ['test_accuracy', 'param_count'],
    ascending=[False, True],
).reset_index(drop=True)
